# Rule-Sensitive Disagreement — testing Property 3.3.3

> **Note:** this run uses `gpt5.2_correct_definition_run2/submission.jsonl`, produced with the corrected
> TAPE/TAUS definitions (`context_map` -> TAPE_v2/TAUS_v2), fixing the earlier bug where TAPE and TAUS
> reused TACO's argument definition.

Property 3.3.3 states: if two instances contain the same sentence under different dataset rules whose gold
labels disagree,

$$R_{\mathrm{arg}}^{(d_i)} \neq R_{\mathrm{arg}}^{(d_j)} \;\Longrightarrow\; f_\theta^{(d_i)} \neq f_\theta^{(d_j)}.$$

TACO/TAPE/TAUS are the same 340 sentences with the same context $(x,c)$; only the guideline differs. This gives
us the exact population the property talks about: sentences where the gold label disagrees between two rules
(238 such sentences across the three rule pairs).

**The test.** On these 238 sentences, a fully rule-blind system (one fixed answer per sentence) always scores
0 % on "correct under both rules" — the two rules disagree by construction, so a single answer can match at most
one side. An unbiased random 50/50 guesser, by contrast, gets *both* sides right with probability
$0.5 \times 0.5 = 25\,\%$ purely by chance, since each side is an independent binary guess. This gives us a
simple, assumption-light benchmark: **is the system's rate above or below random chance?**

In [1]:
import json
from pathlib import Path

BASE = Path('/Users/hanneswidera/Privat/Uni/Master/thesis/GAIC-Thesis')
SUB  = BASE / 'submissions/gpt5.2_correct_definition_run2/submission.jsonl'
LAB  = BASE / 'data/GAIC-2026/data/test_labels.jsonl'
IDX  = range(1, 341)
PAIRS = [('TACO', 'TAPE'), ('TACO', 'TAUS'), ('TAUS', 'TAPE')]

def load(p): return {json.loads(l)['id']: json.loads(l)['label'] for l in open(p)}
gold, pred = load(LAB), load(SUB)
R = {d: {i: gold[f'{d}-test-{i}'] for i in IDX} for d in {'TACO', 'TAPE', 'TAUS'}}
F = {d: {i: pred[f'{d}-test-{i}'] for i in IDX} for d in {'TACO', 'TAPE', 'TAUS'}}

# For each rule pair and pooled across all three: on the sentences where gold disagrees,
# how often is the system correct under both rules, vs. the 25% an unbiased 50/50 guess
# would achieve by chance on the same sentences (0.5 x 0.5, independent per side).
n_all = correct_all = 0
for a, b in PAIRS:
    disagree = [i for i in IDX if R[a][i] != R[b][i]]
    n = len(disagree)
    correct = sum(1 for i in disagree if F[a][i] == R[a][i] and F[b][i] == R[b][i])
    print(f'{a}-{b}: n={n:3d}  correct under both rules={correct:2d} ({correct/n:.1%})')
    n_all += n; correct_all += correct

print(f'\nAll pairs pooled: n={n_all}  correct under both rules={correct_all} ({correct_all/n_all:.1%})')
print(f'Chance baseline (unbiased random guess): 25%  ->  expected {0.25*n_all:.0f} / {n_all}')

TACO-TAPE: n=106  correct under both rules=18 (17.0%)
TACO-TAUS: n= 94  correct under both rules=14 (14.9%)
TAUS-TAPE: n= 38  correct under both rules= 4 (10.5%)

All pairs pooled: n=238  correct under both rules=36 (15.1%)
Chance baseline (unbiased random guess): 25%  ->  expected 60 / 238


All three rule pairs tell the same story. Take TACO–TAPE as a concrete example: of the 340 tweets, the gold
label disagrees between the two guidelines on 106 of them — these are exactly the sentences on which Property
3.3.3 requires the system to change its prediction. The system gets both sides right on only 18 of these 106
sentences (17.0 %). The other two pairs land in the same range: TACO–TAUS at 14 of 94 (14.9 %), and TAUS–TAPE at
4 of 38 (10.5 %). Pooled across all 238 disagreement sentences, the system is correct under both rules 36 times
(15.1 %).

For comparison, an unbiased system that simply guessed `Argument` or `No-Argument` at random on each side would
get both sides right about 25 % of the time by chance alone (a 50/50 guess independently on each side). Every
one of the three rule pairs falls below this chance level, not just the pooled total. A system that genuinely
tracked the applicable guideline would score well above 25 %; a system that ignored the guideline entirely and
answered every sentence the same way both times would score 0 %, since the two guidelines disagree by
construction on these sentences. The observed rate sits closer to the rule-blind extreme than to chance, on
every rule pair individually and in aggregate. Compared to the original run (where TAPE and TAUS mistakenly
reused TACO's definition), correcting the definitions raises the pooled correct-under-both-rules rate from
11.8 % to 15.1 % — a real improvement, but the system is still far below the 25 % chance baseline.

## F1 on agreement-only sentences

As a complement to the disagreement analysis above: for each rule pair, restrict to the sentences where the
two rules' gold labels *agree* (i.e. drop the disagreement sentences from that pair), and compute each
dataset's own F1 (positive class = `Argument`) on that restricted subset. This isolates how well the system
performs on the "easy" shared cases, separate from the rule-sensitive edge cases analyzed above.

In [2]:
from sklearn.metrics import f1_score

POS = 'Argument'

def f1_on(d, idxs):
    y_true = [R[d][i] for i in idxs]
    y_pred = [F[d][i] for i in idxs]
    return f1_score(y_true, y_pred, pos_label=POS, average='binary')

for a, b in PAIRS:
    agree = [i for i in IDX if R[a][i] == R[b][i]]
    n = len(agree)
    f1_a = f1_on(a, agree)
    f1_b = f1_on(b, agree)
    print(f'{a}-{b}: n_agree={n:3d}   {a} F1={f1_a:.3f}   {b} F1={f1_b:.3f}')


TACO-TAPE: n_agree=234   TACO F1=0.920   TAPE F1=0.937
TACO-TAUS: n_agree=246   TACO F1=0.925   TAUS F1=0.920
TAUS-TAPE: n_agree=302   TAUS F1=0.758   TAPE F1=0.783


## Main score: consensus vs non consensus sentences

The Main score averages macro F1 (Argument and No Argument) over TACO, TAPE and TAUS on all 340 sentences.
Split the 340 sentences into the 221 where all three rules agree on the gold label (consensus) and the 119
where at least one rule disagrees (non consensus), and recompute Main separately on each subset using the
same macro F1 definition as the official metric. This turns "if we handled the disagreement sentences almost
as well as the consensus ones, Main would be above 0.9" into a measured number instead of an extrapolation.

In [3]:
def f1_macro(d, idxs):
    y_true = [R[d][i] for i in idxs]
    y_pred = [F[d][i] for i in idxs]
    return f1_score(y_true, y_pred, average='macro')

consensus    = [i for i in IDX if R['TACO'][i] == R['TAPE'][i] == R['TAUS'][i]]
nonconsensus = [i for i in IDX if i not in consensus]

overall, cons, noncons = {}, {}, {}
for d in ('TACO', 'TAPE', 'TAUS'):
    overall[d]  = f1_macro(d, list(IDX))
    cons[d]     = f1_macro(d, consensus)
    noncons[d]  = f1_macro(d, nonconsensus)

main_overall  = sum(overall.values()) / 3
main_cons     = sum(cons.values()) / 3
main_noncons  = sum(noncons.values()) / 3
n_c, n_n = len(consensus), len(nonconsensus)

print(f'n consensus={n_c}   n non consensus={n_n}\n')
for d in ('TACO', 'TAPE', 'TAUS'):
    print(f'{d}: overall={overall[d]:.4f}   consensus={cons[d]:.4f}   non consensus={noncons[d]:.4f}')

print(f'\nMain overall            = {main_overall:.4f}')
print(f'Main consensus only     = {main_cons:.4f}')
print(f'Main non consensus only = {main_noncons:.4f}')
print(f'Weighted blend check    = {(n_c*main_cons + n_n*main_noncons)/(n_c+n_n):.4f}')

n consensus=221   n non consensus=119

TACO: overall=0.8557   consensus=0.9421   non consensus=0.6398
TAPE: overall=0.7833   consensus=0.9518   non consensus=0.4796
TAUS: overall=0.7870   consensus=0.9368   non consensus=0.5205

Main overall            = 0.8087
Main consensus only     = 0.9436
Main non consensus only = 0.5466
Weighted blend check    = 0.8046


In [4]:
latex = []
latex.append(r'\begin{table}[ht]')
latex.append(r'\centering')
latex.append(r'\begin{tabular}{lccc}')
latex.append(r'\toprule')
latex.append(r'Dataset & Overall (n=%d) & Consensus (n=%d) & Non-consensus (n=%d) \\' % (len(IDX), n_c, n_n))
latex.append(r'\midrule')
for d in ('TACO', 'TAPE', 'TAUS'):
    latex.append(rf'{d} & {overall[d]:.3f} & {cons[d]:.3f} & {noncons[d]:.3f} \\')
latex.append(r'\midrule')
latex.append(rf'\textbf{{Main}} & \textbf{{{main_overall:.3f}}} & \textbf{{{main_cons:.3f}}} & \textbf{{{main_noncons:.3f}}} \\')
latex.append(r'\bottomrule')
latex.append(r'\end{tabular}')
latex.append(r'\caption{Macro F1 per dataset on all 340 sentences, on the 221 sentences where TACO, TAPE and TAUS agree on the gold label (consensus), and on the 119 sentences where they disagree (non-consensus).}')
latex.append(r'\label{tab:rule-sensitive-consensus}')
latex.append(r'\end{table}')
print('\n'.join(latex))

\begin{table}[ht]
\centering
\begin{tabular}{lccc}
\toprule
Dataset & Overall (n=340) & Consensus (n=221) & Non-consensus (n=119) \\
\midrule
TACO & 0.856 & 0.942 & 0.640 \\
TAPE & 0.783 & 0.952 & 0.480 \\
TAUS & 0.787 & 0.937 & 0.520 \\
\midrule
\textbf{Main} & \textbf{0.809} & \textbf{0.944} & \textbf{0.547} \\
\bottomrule
\end{tabular}
\caption{Macro F1 per dataset on all 340 sentences, on the 221 sentences where TACO, TAPE and TAUS agree on the gold label (consensus), and on the 119 sentences where they disagree (non-consensus).}
\label{tab:rule-sensitive-consensus}
\end{table}


## Why the system fails on non-consensus but not on consensus

Extract 10 mislabeled non-consensus examples per dataset, check the direction of the errors
(which class gold has vs. which class the system predicted), and check how often the system
gives the *same* prediction across all three rules for a given sentence — on consensus vs.
non-consensus. If the system were rule-sensitive, sameness should track the gold rather than
sit constant.

In [5]:
from collections import Counter

TEST = BASE / 'data/GAIC-2026/data/test.jsonl'
sentence = {}
with open(TEST) as f:
    for line in f:
        row = json.loads(line)
        sentence[row['id']] = row['sentence']

# 10 mislabeled non-consensus examples per dataset, written to a markdown file
lines = ["# Non-consensus mislabeled samples (10 per dataset)", "",
         f"From the {n_n} sentences where TACO, TAPE and TAUS gold labels do not all agree, "
         f"the following are cases where the system's own prediction for that dataset's rule "
         f"differs from that dataset's gold label.", ""]

mislabeled = {}
for d in ('TACO', 'TAPE', 'TAUS'):
    mis = [i for i in nonconsensus if F[d][i] != R[d][i]]
    mislabeled[d] = mis
    lines.append(f"## {d} ({len(mis)} mislabeled out of {n_n} non-consensus sentences)")
    lines.append("")
    for i in mis[:10]:
        oid = f"{d}-test-{i}"
        other = {dd: R[dd][i] for dd in ('TACO', 'TAPE', 'TAUS') if dd != d}
        lines.append(f"### {oid}")
        lines.append(f"- sentence: {sentence[oid]}")
        lines.append(f"- {d} gold: **{R[d][i]}**, {d} pred: **{F[d][i]}**")
        lines.append("- other rules' gold on same sentence: " + ", ".join(f"{dd}={other[dd]}" for dd in other))
        lines.append("")

out_path = BASE / 'submissions/gpt5.2_correct_definition_run2/nonconsensus_mislabeled_examples.md'
out_path.write_text("\n".join(lines))
print(f"wrote {out_path}")

# Error direction per dataset, on non-consensus mislabeled cases
print()
for d in ('TACO', 'TAPE', 'TAUS'):
    dirs = Counter((R[d][i], F[d][i]) for i in mislabeled[d])
    print(f'{d} error directions (gold -> pred): {dict(dirs)}')

print()
for d in ('TACO', 'TAPE', 'TAUS'):
    predc = Counter(F[d][i] for i in nonconsensus)
    goldc = Counter(R[d][i] for i in nonconsensus)
    print(f'{d}: pred dist on non-consensus={dict(predc)}   gold dist on non-consensus={dict(goldc)}')

# Rule-sensitivity check: does the system give the SAME prediction across all three rules
# for a given sentence, regardless of which guideline applies?
same_pred_noncons = sum(1 for i in nonconsensus if F['TACO'][i] == F['TAPE'][i] == F['TAUS'][i])
same_pred_cons    = sum(1 for i in consensus    if F['TACO'][i] == F['TAPE'][i] == F['TAUS'][i])
print(f'\nSame prediction across all 3 rules -- consensus:     {same_pred_cons}/{n_c} ({same_pred_cons/n_c:.1%})')
print(f'Same prediction across all 3 rules -- non-consensus: {same_pred_noncons}/{n_n} ({same_pred_noncons/n_n:.1%})')

wrote /Users/hanneswidera/Privat/Uni/Master/thesis/GAIC-Thesis/submissions/gpt5.2_correct_definition_run2/nonconsensus_mislabeled_examples.md

TACO error directions (gold -> pred): {('Argument', 'No-Argument'): 25, ('No-Argument', 'Argument'): 12}
TAPE error directions (gold -> pred): {('No-Argument', 'Argument'): 53, ('Argument', 'No-Argument'): 7}
TAUS error directions (gold -> pred): {('No-Argument', 'Argument'): 46, ('Argument', 'No-Argument'): 11}

TACO: pred dist on non-consensus={'Argument': 75, 'No-Argument': 44}   gold dist on non-consensus={'Argument': 88, 'No-Argument': 31}
TAPE: pred dist on non-consensus={'Argument': 72, 'No-Argument': 47}   gold dist on non-consensus={'No-Argument': 93, 'Argument': 26}
TAUS: pred dist on non-consensus={'Argument': 75, 'No-Argument': 44}   gold dist on non-consensus={'No-Argument': 79, 'Argument': 40}

Same prediction across all 3 rules -- consensus:     206/221 (93.2%)
Same prediction across all 3 rules -- non-consensus: 82/119 (68.9%)


## Failure examples

For each rule pair, 4 concrete examples where the system does *not* get both sides right on a
disagreement sentence (i.e. it either matches only one rule's gold label, or matches neither).
This makes the failure mode in the numbers above legible: what the sentence actually looks like,
what each guideline says the gold label is, and what the system predicted for each.

In [6]:
TEST = BASE / 'data/GAIC-2026/data/test.jsonl'
sentence = {}
with open(TEST) as f:
    for line in f:
        row = json.loads(line)
        sentence[row['id']] = row['sentence']

N_EXAMPLES = 4

for a, b in PAIRS:
    disagree = [i for i in IDX if R[a][i] != R[b][i]]
    failures = [i for i in disagree if not (F[a][i] == R[a][i] and F[b][i] == R[b][i])]
    print(f'=== {a}-{b}  ({len(failures)} failures out of {len(disagree)} disagreements) ===\n')
    for i in failures[:N_EXAMPLES]:
        print(f'sentence: {sentence[f"{a}-test-{i}"]}')
        print(f'  {a}  gold={R[a][i]:<11} pred={F[a][i]}')
        print(f'  {b}  gold={R[b][i]:<11} pred={F[b][i]}')
        print()
    print()

=== TACO-TAPE  (88 failures out of 106 disagreements) ===

sentence: @Cameron_Parker @Barbara_Dudley @Shannon_Walker @Veronica_Simpson @Jamie_Walton @Alyssa_Long So if you are going to quote him, quote everything he had to say about it!
  TACO  gold=Argument    pred=Argument
  TAPE  gold=No-Argument pred=Argument

sentence: @Jessica_Gross It's OK to not like things (but don't be a dick about it)
  TACO  gold=Argument    pred=Argument
  TAPE  gold=No-Argument pred=Argument

sentence: @Amber_Wright @Mary_Escobar DWP is going to be busy!
  TACO  gold=Argument    pred=Argument
  TAPE  gold=No-Argument pred=Argument

sentence: @Barbara_Dudley @David_Davis Show me the descriptions of skin colors in Tolkien’s writing.
  TACO  gold=Argument    pred=Argument
  TAPE  gold=No-Argument pred=Argument


=== TACO-TAUS  (80 failures out of 94 disagreements) ===

sentence: @Cameron_Parker @Barbara_Dudley @Shannon_Walker @Veronica_Simpson @Jamie_Walton @Alyssa_Long So if you are going to quote him, quot